---
**Study 04: Full Fine-Tune (20 Epochs)**  
Train longer with v3 config (FocalDiceLoss + stratified sampler w=3).
---

## 1. Setup & GPU Check

In [ ]:
# pyarrow TxF workaround (must be set before any sklearn import)
import os; os.environ["PYARROW_IGNORE_ZERO_COPY"] = "1"
import sklearn

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
sys.path.insert(0, str(Path.cwd().parent))

import torch
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 2. Configuration

In [ ]:
EPOCHS = 20; BATCH_SIZE = 4; LR = 1e-3; SEED = 42
TUMOR_WEIGHT = 3.0
OUTPUT_DIR = Path('../models/s9_finetune_v4')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output: {OUTPUT_DIR}\nEpochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR}, Seed: {SEED}, TW={TUMOR_WEIGHT}')
from src.utils import set_seed; set_seed(SEED)


## 3. Data Preparation

In [ ]:
from src.config import DEVICE, print_device_info; print_device_info()
from src.data_loader import DatasetConfig, DataPathManager, VolumeWiseSplitter, create_2d_dataloaders
from src.preprocessing import PreprocessingTransform, AugmentedPreprocessingTransform, CLAHEProcessor

path_manager = DataPathManager(); volume_index = path_manager.build_index()
splitter = VolumeWiseSplitter(); splits = splitter.load_splits(DatasetConfig.SPLITS_DIR)
print(f'Train: {len(splits["train"])} vols, Val: {len(splits["val"])} vols, Test: {len(splits["test"])} vols')

clahe = CLAHEProcessor(clip=2.0, grid=(8,8))
transform_train = AugmentedPreprocessingTransform((256, 256), -100, 400, clahe)
transform_val = PreprocessingTransform((256, 256), -100, 400)
train_loader, val_loader, test_loader = create_2d_dataloaders(
    volume_index, splits['train'], splits['val'], splits['test'],
    batch_size=BATCH_SIZE,
    transform_train=transform_train, transform_val=transform_val,
    use_tumor_sampler=True, tumor_sampler_weight=TUMOR_WEIGHT)
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}')

## 4. Model & Trainer Setup

In [ ]:
from src.models import create_model, count_params
from src.trainer import Trainer
from src.config import PHASE4_RESEARCH_CONFIG

model = create_model('mobilenetv2_unet', in_channels=1, out_channels=1, pretrained=True).to(DEVICE)
print(f'Parameters: {count_params(model):,}')

train_config = dict(PHASE4_RESEARCH_CONFIG)
train_config['use_focal_dice'] = True
train_config['focal_alpha'] = 0.75
trainer = Trainer(model=model, train_loader=train_loader, val_loader=val_loader,
    config=train_config, learning_rate=LR, num_epochs=EPOCHS,
    mixed_precision=True, output_dir=str(OUTPUT_DIR))
print('Trainer initialized with FocalDiceLoss + stratified sampler')


## 5. Training (20 epochs)

In [ ]:
print('Starting 20-epoch fine-tune...')
trainer.fit(epochs=EPOCHS, patience=10)
best_dice = trainer.best_val_dice
print(f'Best val dice: {best_dice:.4f}')
print(f'Best at epoch: {1+trainer.history["val_dice"].index(max(trainer.history["val_dice"]))}')

## 6. Training Curves

In [ ]:
hist = trainer.history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(hist['val_dice'], 'r-o', label='Val Dice')
axes[0].plot(hist['val_iou'], 'g-o', label='Val IoU')
axes[0].plot(hist['val_tumor_dice'], 'm-o', label='Val Tumor Dice')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Score'); axes[0].set_title('Validation Metrics')
axes[0].legend(); axes[0].grid(True)
axes[1].plot(hist['train_loss'], 'b-o', label='Train Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].set_title('Training Loss')
axes[1].legend(); axes[1].grid(True)
axes[2].plot(hist['val_auprc'], 'c-o', label='AUPRC')
axes[2].plot(hist['val_fg_frac'], 'y-o', label='FG%')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Value'); axes[2].set_title('Diagnostics')
axes[2].legend(); axes[2].grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Test Set Evaluation
> Runs `evaluate()` inside the notebook (pyarrow workaround applied in cell 1).

In [ ]:
test_metrics = trainer.evaluate(test_loader)
print(f'Test Dice:       {test_metrics["dice"]:.4f}')
print(f'Test IoU:        {test_metrics["iou"]:.4f}')
print(f'Tumor-only Dice: {test_metrics["tumor_only_dice"]:.4f}')
print(f'Tumor-only IoU:  {test_metrics["tumor_only_iou"]:.4f}')
print(f'Precision:       {test_metrics["precision"]:.4f}')
print(f'Recall:          {test_metrics["recall"]:.4f}')
print(f'Pred FG frac:    {test_metrics["pred_fg_fraction"]:.4f}')
print(f'AUROC:           {test_metrics["auroc"]:.4f}')
print(f'AUPRC:           {test_metrics["auprc"]:.4f}')

with open(OUTPUT_DIR / 'history.json', 'w') as f:
    json.dump({k: [float(x) for x in v] for k, v in hist.items()}, f, indent=2)
with open(OUTPUT_DIR / 'test_metrics.json', 'w') as f:
    tm = {k: float(v) for k, v in test_metrics.items()}
    json.dump(tm, f, indent=2)
print('\nResults saved to history.json / test_metrics.json')

## 8. Summary

In [ ]:
hist = trainer.history
print('='*60)
print('FINETUNE TRAINING SUMMARY')
print('='*60)
print(f'Epochs: {EPOCHS}')
print(f'Loss: FocalDiceLoss (focal_alpha=0.75, gamma=2.0)')
print(f'Sampler: stratified (tumor_weight={TUMOR_WEIGHT})')
print(f'Best val dice:       {trainer.best_val_dice:.4f}')
print(f'Best val tumor dice: {max(hist["val_tumor_dice"]):.4f}')
print(f'Test dice:           {test_metrics["dice"]:.4f}')
print(f'Test tumor-only dice: {test_metrics["tumor_only_dice"]:.4f}')
print(f'Precision:  {test_metrics["precision"]:.4f}')
print(f'Recall:     {test_metrics["recall"]:.4f}')
print(f'Pred FG%:   {test_metrics["pred_fg_fraction"]*100:.2f}%')
print(f'AUROC:      {test_metrics["auroc"]:.4f}')
print(f'AUPRC:      {test_metrics["auprc"]:.4f}')
if hist.get('val_tumor_dice'):
    trend = hist['val_tumor_dice']
    print(f'\nTumor Dice trend: {" → ".join(f"{v:.4f}" for v in trend)}')
    if len(trend) >= 2 and trend[-1] > trend[0]:
        print('+ Tumor Dice is improving — more epochs may help further')
    elif len(trend) >= 2 and trend[-1] < trend[0]/2:
        print('! Tumor Dice decreasing — may need a different strategy')
    else:
        print('~ Tumor Dice relatively stable')
print(f'\nCompare with v3 (5-epoch): Dice=0.614, T Dice=0.018, AUPRC=0.260')

---
**Note:** If cell 7 crashes with pyarrow, run `tracking/_eval_v3.py` from `C:\Users\alanm\AppData\Local\Temp\opencode` instead (set `MODEL_DIR` to `s9_finetune_v4`).